In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

from morgan_fingerprints import generate_morgan_features

In [2]:
# The five datasets are processed one by one
files = [
    "AlmanacLINCS_PRISM.csv",
    "DrugCombDBLINCS_PRISM.csv",
    "OncologyScreenLINCS_PRISM.csv",
    "OneilLINCS_PRISM.csv"
]

all_results = []

In [3]:
from sklearn.ensemble import AdaBoostRegressor

# Loop through datasets: train, evaluate, print metrics, then continue
for file in files:
    print("\\n" + "=" * 60)
    print("Dataset:", file)
    print("=" * 60)

    if not os.path.exists(file):
        print("File not found - skipping.")
        continue

    data = pd.read_csv(file)
    required = ["drugname1", "drugname2", "cellline", "score"]
    if not all(col in data.columns for col in required):
        print("Required columns missing - skipping.")
        continue

    data = data[required].dropna().drop_duplicates().copy()
    data["score"] = pd.to_numeric(data["score"], errors="coerce")
    data = data.dropna(subset=["score"])

    if len(data) < 5:
        print("Not enough valid rows to train/test - skipping.")
        continue

    # Create features for this dataset only
    X = generate_morgan_features(
        data,
        drug1_col="drugname1",
        drug2_col="drugname2",
        radius=2,
        n_bits=512
    )
    X["cellline"] = data["cellline"].astype(str).values
    y = data["score"].astype(float)

    # Split this dataset into 80% training and 20% testing
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    fp_columns = [col for col in X.columns if col != "cellline"]
    prepare_data = ColumnTransformer([
        ("fingerprints", SimpleImputer(strategy="constant", fill_value=np.uint8(0)), fp_columns),
        ("cellline", OneHotEncoder(handle_unknown="ignore"), ["cellline"])
    ])

    # Create a fresh model for this dataset
    model = make_pipeline(prepare_data, AdaBoostRegressor(n_estimators=50, random_state=42))
    model.fit(X_train, y_train)

    # Evaluate before moving to the next dataset
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    print("Rows used:", len(data))
    print("MAE:", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R2:", round(r2, 4))

    all_results.append({
        "Dataset": file,
        "Rows": len(data),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

# Show and save a final summary after all datasets have been processed
results = pd.DataFrame(all_results)
print("\\nFINAL SUMMARY")
display(results)
results.to_csv("adaboost_separate_dataset_metrics.csv", index=False)
print("Metrics saved to adaboost_separate_dataset_metrics.csv")

\n============================================================
Dataset: AlmanacLINCS_PRISM.csv
Rows used: 35512
MAE: 37.7587
RMSE: 50.9045
R2: 0.1308
\n============================================================
Dataset: DrugCombDBLINCS_PRISM.csv
Rows used: 17144
MAE: 4.3871
RMSE: 6.0332
R2: -0.0032
\n============================================================
Dataset: OncologyScreenLINCS_PRISM.csv
Rows used: 1160
MAE: 16.2016
RMSE: 23.0375
R2: 0.0848
\n============================================================
Dataset: OneilLINCS_PRISM.csv
Rows used: 2714
MAE: 14.3472
RMSE: 18.9873
R2: 0.2802
\nFINAL SUMMARY


,Dataset,Rows,MAE,RMSE,R2
0,AlmanacLINCS_PRISM.csv,35512,37.758660,50.904490,0.130759
1,DrugCombDBLINCS_PRISM.csv,17144,4.387116,6.033198,-0.003173
2,OncologyScreenLINCS_PRISM.csv,1160,16.201568,23.037536,0.084789
3,OneilLINCS_PRISM.csv,2714,14.347242,18.987264,0.280156


Metrics saved to adaboost_separate_dataset_metrics.csv
